In [4]:
import pandas as pd

In [5]:
trans_clean = pd.read_csv("transactions_cleaned.csv")

In [11]:
trans_clean['transaction_date'] = pd.to_datetime(trans_clean['transaction_date'])
trans_clean['cohort_month'] = pd.to_datetime(trans_clean['cohort_month'].astype(str))

In [16]:
trans_clean['transaction_month'] = (
    trans_clean['transaction_date']
    .dt.to_period('M')
    .dt.to_timestamp()
)

print(trans_clean.head())

  transaction_id user_id transaction_date product_category      product_name  \
0        T000001  U00001       2024-11-27     Subscription           Starter   
1        T000002  U00001       2024-11-29         Software           Starter   
2        T000003  U00001       2025-04-25     Subscription        Enterprise   
3        T000004  U00001       2025-05-24      Accessories           Starter   
4        T000005  U00001       2025-05-23           Course  Analytics Add-on   

   quantity  discount payment_method first_transaction_date cohort_month  \
0         2         0           Card             2024-11-27   2024-11-01   
1         3        15    Net Banking             2024-11-27   2024-11-01   
2         2        15            UPI             2024-11-27   2024-11-01   
3         1        15         PayPal             2024-11-27   2024-11-01   
4         2         0           Card             2024-11-27   2024-11-01   

  transaction_month  
0        2024-11-01  
1        2024-11-0

In [17]:
trans_clean['cohort_index'] = (
    (trans_clean['transaction_month'].dt.year - trans_clean['cohort_month'].dt.year) * 12
    +
    (trans_clean['transaction_month'].dt.month - trans_clean['cohort_month'].dt.month)
)
print(trans_clean.head())

  transaction_id user_id transaction_date product_category      product_name  \
0        T000001  U00001       2024-11-27     Subscription           Starter   
1        T000002  U00001       2024-11-29         Software           Starter   
2        T000003  U00001       2025-04-25     Subscription        Enterprise   
3        T000004  U00001       2025-05-24      Accessories           Starter   
4        T000005  U00001       2025-05-23           Course  Analytics Add-on   

   quantity  discount payment_method first_transaction_date cohort_month  \
0         2         0           Card             2024-11-27   2024-11-01   
1         3        15    Net Banking             2024-11-27   2024-11-01   
2         2        15            UPI             2024-11-27   2024-11-01   
3         1        15         PayPal             2024-11-27   2024-11-01   
4         2         0           Card             2024-11-27   2024-11-01   

  transaction_month  cohort_index  
0        2024-11-01       

In [18]:
cohort_data = (
    trans_clean.groupby(['cohort_month', 'cohort_index'])['user_id']
    .nunique()
    .reset_index()
)

print(cohort_data.to_string(index=False))

cohort_month  cohort_index  user_id
  2024-01-01             0      450
  2024-01-01             1      201
  2024-01-01             2      164
  2024-01-01             3      139
  2024-01-01             4      107
  2024-01-01             5       85
  2024-01-01             6       75
  2024-01-01             7       69
  2024-01-01             8       46
  2024-01-01             9       39
  2024-01-01            10       32
  2024-01-01            11       20
  2024-02-01             0      609
  2024-02-01             1      278
  2024-02-01             2      234
  2024-02-01             3      195
  2024-02-01             4      151
  2024-02-01             5      131
  2024-02-01             6      102
  2024-02-01             7       82
  2024-02-01             8       69
  2024-02-01             9       46
  2024-02-01            10       52
  2024-02-01            11       18
  2024-03-01             0      714
  2024-03-01             1      326
  2024-03-01             2  

In [19]:
retention_matrix = cohort_data.pivot_table(
    index='cohort_month',
    columns='cohort_index',
    values='user_id'
)

retention_matrix.to_csv('cohort_retention_matrix.csv', index=True)
print(retention_matrix.to_string(index=True))

cohort_index     0      1      2      3      4      5      6      7     8     9     10    11   12
cohort_month                                                                                     
2024-01-01    450.0  201.0  164.0  139.0  107.0   85.0   75.0   69.0  46.0  39.0  32.0  20.0  NaN
2024-02-01    609.0  278.0  234.0  195.0  151.0  131.0  102.0   82.0  69.0  46.0  52.0  18.0  NaN
2024-03-01    714.0  326.0  278.0  208.0  177.0  162.0  109.0   87.0  68.0  51.0  45.0  24.0  NaN
2024-04-01    683.0  310.0  273.0  222.0  190.0  150.0  127.0  103.0  80.0  58.0  39.0  27.0  NaN
2024-05-01    735.0  341.0  293.0  235.0  177.0  155.0  108.0  103.0  76.0  60.0  45.0  27.0  NaN
2024-06-01    721.0  360.0  292.0  242.0  197.0  162.0  118.0  107.0  82.0  74.0  53.0  34.0  NaN
2024-07-01    736.0  351.0  282.0  235.0  179.0  150.0  140.0  107.0  90.0  66.0  48.0  24.0  1.0
2024-08-01    815.0  351.0  315.0  255.0  217.0  184.0  131.0  124.0  91.0  82.0  61.0  28.0  NaN
2024-09-01    654.0 

In [2]:
retention_percentage = (
    retention_matrix
    .div(retention_matrix.iloc[:,0], axis=0)
    * 100
)

retention_percentage.to_csv('cohort_retention_percentage.csv', index=True)

NameError: name 'retention_matrix' is not defined